In [ ]:
# ============================================================
# Self-Calibrating Embedded UAV Vision for Illumination-Robust
# Target Recognition in 5G/6G Aerial Cyber-Physical Systems
#
# Stable Kaggle Code:
# - Single T4 GPU
# - No DataParallel
# - IID + Cross-Illumination + LOIO tests
# - Illumination-specific calibration
# - Reliability-guided safe abstention
# - 600 dpi figures
# - Excel paper-ready tables
# ============================================================

!pip -q install scikit-learn pandas matplotlib openpyxl

import os
import re
import time
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision as tv
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ============================================================
# 1. Reproducibility and runtime setup
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = True if DEVICE.type == "cuda" else False

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU used:", torch.cuda.get_device_name(0))
    print("GPU count available:", torch.cuda.device_count())

# Important: do not use DataParallel on Kaggle T4 x2
USE_MULTIGPU = False

# ============================================================
# 2. Output folders
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/self_calibrating_uav_6g_results")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"

for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# 3. Experiment configuration
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

EXPECTED_COLORS = [
    "Black", "Blue", "Gray", "Orange", "Pink",
    "Purple", "Skyblue", "White", "Yellow"
]

EXPECTED_ILLUMS = [
    "fluorescentLight", "indoor", "indoorNight", "sunLight"
]

# Stable settings for T4
IMG_SIZE = 160
BATCH_SIZE = 96
EPOCHS = 5
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PIN_MEMORY = True if DEVICE.type == "cuda" else False

# Set True for quick IID only.
# Set False for full conference paper experiments.
FAST_MODE = False

CONF_THRESHOLDS = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90]
COVERAGE_LEVELS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

# ============================================================
# 4. Print Kaggle input structure
# ============================================================

def print_kaggle_tree(root="/kaggle/input", max_depth=3):
    print("=" * 90)
    print("KAGGLE INPUT DIRECTORY STRUCTURE")
    print("=" * 90)

    root = Path(root)

    for path, dirs, files in os.walk(root):
        depth = len(Path(path).relative_to(root).parts)

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "  " * depth
        print(f"{indent}{Path(path).name}/")

        for f in files[:6]:
            print(f"{indent}  {f}")

        if len(files) > 6:
            print(f"{indent}  ... {len(files) - 6} more files")

print_kaggle_tree()

# ============================================================
# 5. Dataset scanner
# ============================================================

def normalize_token(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

COLOR_MAP = {normalize_token(c): c for c in EXPECTED_COLORS}
ILLUM_MAP = {normalize_token(i): i for i in EXPECTED_ILLUMS}

def infer_color_illum_from_path(path):
    parts = [normalize_token(p) for p in Path(path).parts]

    color = None
    illum = None

    for p in parts:
        if p in COLOR_MAP:
            color = COLOR_MAP[p]
        if p in ILLUM_MAP:
            illum = ILLUM_MAP[p]

    return color, illum

def scan_dataset(input_root="/kaggle/input"):
    rows = []

    for f in Path(input_root).rglob("*"):
        if f.suffix.lower() in IMG_EXTS:
            color, illum = infer_color_illum_from_path(f)

            if color is not None and illum is not None:
                rows.append({
                    "filepath": str(f),
                    "color": color,
                    "illumination": illum
                })

    data = pd.DataFrame(rows)

    if data.empty:
        raise RuntimeError(
            "No valid images detected. Please check folder names or update EXPECTED_COLORS / EXPECTED_ILLUMS."
        )

    data["label_name"] = data["color"]
    data["domain_name"] = data["illumination"]

    return data

df = scan_dataset()

classes = sorted(df["label_name"].unique())
illums = sorted(df["domain_name"].unique())

class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

illum_to_idx = {d: i for i, d in enumerate(illums)}
idx_to_illum = {i: d for d, i in illum_to_idx.items()}

df["label"] = df["label_name"].map(class_to_idx)
df["illum_id"] = df["domain_name"].map(illum_to_idx)
df["strata"] = df["label_name"] + "_" + df["domain_name"]

print("\nDetected images:", len(df))
print("\nClass mapping:", class_to_idx)
print("Illumination mapping:", illum_to_idx)

print("\nColor distribution:")
print(df["color"].value_counts())

print("\nIllumination distribution:")
print(df["illumination"].value_counts())

df.to_csv(OUTPUT_DIR / "dataset_manifest.csv", index=False)

# ============================================================
# 6. Transforms
# ============================================================

train_tfms = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.12,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# 7. Dataset and DataLoader
# ============================================================

class ColorIllumDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        try:
            img = Image.open(row["filepath"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(0, 0, 0))

        if self.transform:
            img = self.transform(img)

        return {
            "image": img,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "illum": torch.tensor(int(row["illum_id"]), dtype=torch.long),
            "filepath": row["filepath"]
        }

def make_loader(frame, train=False):
    ds = ColorIllumDataset(
        frame,
        transform=train_tfms if train else test_tfms
    )

    if train:
        counts = frame["label"].value_counts().to_dict()
        weights = frame["label"].map(lambda x: 1.0 / counts[x]).values

        sampler = WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True
        )

        loader = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            drop_last=False
        )
    else:
        loader = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
            drop_last=False
        )

    return loader

# ============================================================
# 8. Split protocols
# ============================================================

def safe_stratified_split(data, test_size, stratify_col, seed=SEED):
    counts = data[stratify_col].value_counts()

    if (counts < 2).any():
        return train_test_split(
            data,
            test_size=test_size,
            random_state=seed,
            shuffle=True
        )

    return train_test_split(
        data,
        test_size=test_size,
        random_state=seed,
        stratify=data[stratify_col],
        shuffle=True
    )

def make_iid_split(data):
    train_val, test = safe_stratified_split(
        data,
        test_size=0.20,
        stratify_col="strata"
    )

    train, val = safe_stratified_split(
        train_val,
        test_size=0.20,
        stratify_col="strata"
    )

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True)
    )

def make_cross_illum_split(data):
    train_domains = [d for d in ["indoor", "fluorescentLight"] if d in data["domain_name"].unique()]
    test_domains = [d for d in ["indoorNight", "sunLight"] if d in data["domain_name"].unique()]

    train_val = data[data["domain_name"].isin(train_domains)].copy()
    test = data[data["domain_name"].isin(test_domains)].copy()

    train, val = safe_stratified_split(
        train_val,
        test_size=0.20,
        stratify_col="label_name"
    )

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True)
    )

def make_loio_split(data, heldout_domain):
    train_val = data[data["domain_name"] != heldout_domain].copy()
    test = data[data["domain_name"] == heldout_domain].copy()

    train, val = safe_stratified_split(
        train_val,
        test_size=0.20,
        stratify_col="strata"
    )

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True)
    )

if FAST_MODE:
    protocols = {
        "IID": make_iid_split(df)
    }
else:
    protocols = {
        "IID": make_iid_split(df),
        "Cross_Illumination_Shift": make_cross_illum_split(df),
        "LOIO_sunLight": make_loio_split(df, "sunLight"),
        "LOIO_indoorNight": make_loio_split(df, "indoorNight")
    }

for name, (tr, va, te) in protocols.items():
    print("\n" + "=" * 90)
    print(name)
    print("Train:", tr.shape, tr["domain_name"].value_counts().to_dict())
    print("Val:  ", va.shape, va["domain_name"].value_counts().to_dict())
    print("Test: ", te.shape, te["domain_name"].value_counts().to_dict())

# ============================================================
# 9. Model: Self-Calibrating MobileNetV3-Small
# ============================================================

class SelfCalibratingMobileNetV3(nn.Module):
    def __init__(self, num_classes, num_illums):
        super().__init__()

        try:
            weights = tv.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
            base = tv.models.mobilenet_v3_small(weights=weights)
            print("Loaded ImageNet pretrained MobileNetV3-Small.")
        except Exception as e:
            print("Pretrained weights failed. Using random initialization.")
            print("Reason:", str(e))
            base = tv.models.mobilenet_v3_small(weights=None)

        feat_dim = base.classifier[0].in_features
        base.classifier = nn.Identity()

        self.backbone = base
        self.dropout = nn.Dropout(0.25)

        self.target_head = nn.Linear(feat_dim, num_classes)
        self.illum_head = nn.Linear(feat_dim, num_illums)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)

        target_logits = self.target_head(feat)
        illum_logits = self.illum_head(feat)

        return target_logits, illum_logits

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ============================================================
# 10. Metrics
# ============================================================

def probability_entropy(probs):
    p = np.clip(probs, 1e-12, 1.0)
    return -np.sum(p * np.log(p), axis=1) / np.log(p.shape[1])

def expected_calibration_error(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        low = bins[i]
        high = bins[i + 1]

        mask = (confidences > low) & (confidences <= high)

        if mask.sum() > 0:
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            ece += (mask.sum() / len(labels)) * abs(bin_acc - bin_conf)

    return float(ece)

def brier_score_multiclass(probs, labels):
    y_onehot = np.eye(probs.shape[1])[labels]
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))

def classification_metrics(probs, labels):
    pred = probs.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, pred),
        "balanced_accuracy": balanced_accuracy_score(labels, pred),
        "precision_macro": precision_score(labels, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, pred, average="weighted", zero_division=0),
        "ece": expected_calibration_error(probs, labels),
        "brier": brier_score_multiclass(probs, labels),
        "mean_confidence": float(probs.max(axis=1).mean()),
        "mean_entropy": float(probability_entropy(probs).mean())
    }

# ============================================================
# 11. Training and inference
# ============================================================

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_epoch(model, loader, optimizer, epoch):
    model.train()

    total_loss = 0.0
    total_n = 0
    start = time.time()

    for batch_idx, batch in enumerate(loader, start=1):
        x = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)
        y = batch["label"].to(DEVICE, non_blocking=PIN_MEMORY)
        illum = batch["illum"].to(DEVICE, non_blocking=PIN_MEMORY)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            target_logits, illum_logits = model(x)

            loss_target = F.cross_entropy(target_logits, y)
            loss_illum = F.cross_entropy(illum_logits, illum)

            loss = loss_target + 0.25 * loss_illum

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n += bs

        if batch_idx == 1 or batch_idx % 20 == 0 or batch_idx == len(loader):
            elapsed = (time.time() - start) / 60
            print(
                f"Epoch {epoch} | Batch {batch_idx}/{len(loader)} | "
                f"loss={loss.item():.4f} | elapsed={elapsed:.2f} min"
            )

    return total_loss / max(total_n, 1)

@torch.no_grad()
def collect_outputs(model, loader):
    model.eval()

    target_logits_all = []
    illum_logits_all = []
    labels_all = []
    illums_all = []
    paths_all = []

    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            target_logits, illum_logits = model(x)

        target_logits_all.append(target_logits.detach().cpu())
        illum_logits_all.append(illum_logits.detach().cpu())

        labels_all.extend(batch["label"].numpy().tolist())
        illums_all.extend(batch["illum"].numpy().tolist())
        paths_all.extend(batch["filepath"])

    return {
        "target_logits": torch.cat(target_logits_all, dim=0),
        "illum_logits": torch.cat(illum_logits_all, dim=0),
        "labels": np.array(labels_all),
        "illums": np.array(illums_all),
        "paths": paths_all
    }

# ============================================================
# 12. Temperature scaling
# ============================================================

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        return logits / torch.exp(self.log_temperature)

def fit_temperature(logits, labels, max_iter=80):
    logits = logits.float().to(DEVICE)
    labels = torch.tensor(labels, dtype=torch.long).to(DEVICE)

    temp_model = TemperatureScaler().to(DEVICE)

    optimizer = torch.optim.LBFGS(
        [temp_model.log_temperature],
        lr=0.01,
        max_iter=max_iter
    )

    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(temp_model(logits), labels)
        loss.backward()
        return loss

    optimizer.step(closure)

    T = torch.exp(temp_model.log_temperature).detach().cpu().item()

    if np.isnan(T) or T <= 0:
        T = 1.0

    return float(T)

def fit_illumination_temperatures(val_outputs):
    logits = val_outputs["target_logits"]
    labels = val_outputs["labels"]
    illum_arr = val_outputs["illums"]

    global_T = fit_temperature(logits, labels)

    temp_map = {}

    for illum_id in sorted(np.unique(illum_arr)):
        mask = illum_arr == illum_id

        if mask.sum() >= 20:
            temp_map[int(illum_id)] = fit_temperature(logits[mask], labels[mask])
        else:
            temp_map[int(illum_id)] = global_T

    return global_T, temp_map

def apply_illumination_temperature(outputs, temp_map, default_T):
    logits = outputs["target_logits"].clone()
    scaled = logits.clone()

    for i in range(len(logits)):
        illum_id = int(outputs["illums"][i])
        T = temp_map.get(illum_id, default_T)
        scaled[i] = logits[i] / T

    out = dict(outputs)
    out["target_logits"] = scaled

    return out

# ============================================================
# 13. Reliability, abstention, and protocol interpretation
# ============================================================

def compute_reliability(outputs):
    target_probs = F.softmax(outputs["target_logits"], dim=1).numpy()
    illum_probs = F.softmax(outputs["illum_logits"], dim=1).numpy()

    target_conf = target_probs.max(axis=1)
    illum_conf = illum_probs.max(axis=1)
    entropy = probability_entropy(target_probs)

    reliability = 0.60 * target_conf + 0.25 * (1.0 - entropy) + 0.15 * illum_conf
    reliability = np.clip(reliability, 0.0, 1.0)

    return reliability, target_conf, illum_conf, entropy

def get_protocol_meaning(protocol):
    protocol_meaning = {
        "IID": "Mixed-illumination UAV perception under known operating conditions",
        "Cross_Illumination_Shift": "UAV perception under unseen night/sunlight shift after training on indoor/fluorescent conditions",
        "LOIO_sunLight": "Unseen sunlight deployment condition for outdoor UAV operation",
        "LOIO_indoorNight": "Unseen low-light/night-like deployment condition for UAV operation"
    }

    sixg_relevance = {
        "IID": "Baseline onboard perception reliability before 5G/6G transmission",
        "Cross_Illumination_Shift": "Robustness test for aerial perception under changing environmental conditions",
        "LOIO_sunLight": "Outdoor aerial network deployment relevance",
        "LOIO_indoorNight": "Night/low-light aerial monitoring relevance"
    }

    return protocol_meaning.get(protocol, ""), sixg_relevance.get(protocol, "")

def add_protocol_columns(df_in):
    df_out = df_in.copy()

    meanings = []
    relevances = []

    for p in df_out["protocol"]:
        m, r = get_protocol_meaning(p)
        meanings.append(m)
        relevances.append(r)

    df_out["uav_protocol_meaning"] = meanings
    df_out["sixg_aerial_relevance"] = relevances

    return df_out

def per_illumination_metrics(outputs, protocol_name, method_name):
    probs = F.softmax(outputs["target_logits"], dim=1).numpy()
    labels = outputs["labels"]
    pred = probs.argmax(axis=1)
    illum_arr = outputs["illums"]

    rows = []

    for illum_id in sorted(np.unique(illum_arr)):
        mask = illum_arr == illum_id

        rows.append({
            "protocol": protocol_name,
            "method": method_name,
            "illumination": idx_to_illum[int(illum_id)],
            "n": int(mask.sum()),
            "accuracy": accuracy_score(labels[mask], pred[mask]),
            "balanced_accuracy": balanced_accuracy_score(labels[mask], pred[mask]),
            "precision_macro": precision_score(labels[mask], pred[mask], average="macro", zero_division=0),
            "recall_macro": recall_score(labels[mask], pred[mask], average="macro", zero_division=0),
            "f1_macro": f1_score(labels[mask], pred[mask], average="macro", zero_division=0),
            "ece": expected_calibration_error(probs[mask], labels[mask]),
            "mean_confidence": float(probs[mask].max(axis=1).mean()),
            "mean_entropy": float(probability_entropy(probs[mask]).mean())
        })

    return pd.DataFrame(rows)

def selective_prediction_table(outputs, protocol_name, method_name):
    probs = F.softmax(outputs["target_logits"], dim=1).numpy()
    labels = outputs["labels"]
    pred = probs.argmax(axis=1)

    reliability, _, _, _ = compute_reliability(outputs)

    rows = []

    for thr in CONF_THRESHOLDS:
        keep = reliability >= thr
        coverage = float(keep.mean())

        if keep.sum() == 0:
            acc = np.nan
            f1 = np.nan
            risk = np.nan
        else:
            acc = accuracy_score(labels[keep], pred[keep])
            f1 = f1_score(labels[keep], pred[keep], average="macro", zero_division=0)
            risk = 1.0 - acc

        rows.append({
            "protocol": protocol_name,
            "method": method_name,
            "reliability_threshold": thr,
            "coverage": coverage,
            "abstention_rate": 1.0 - coverage,
            "selective_accuracy": acc,
            "selective_f1_macro": f1,
            "selective_risk": risk,
            "accepted_samples": int(keep.sum()),
            "rejected_samples": int((~keep).sum())
        })

    return pd.DataFrame(rows)

def coverage_control_table(outputs, protocol_name, method_name):
    probs = F.softmax(outputs["target_logits"], dim=1).numpy()
    labels = outputs["labels"]
    pred = probs.argmax(axis=1)

    reliability, _, _, _ = compute_reliability(outputs)

    rows = []

    for cov in COVERAGE_LEVELS:
        threshold = np.quantile(reliability, 1.0 - cov)
        keep = reliability >= threshold

        acc = accuracy_score(labels[keep], pred[keep])
        f1 = f1_score(labels[keep], pred[keep], average="macro", zero_division=0)

        rows.append({
            "protocol": protocol_name,
            "method": method_name,
            "target_coverage": cov,
            "actual_coverage": float(keep.mean()),
            "threshold": float(threshold),
            "selective_accuracy": acc,
            "selective_f1_macro": f1,
            "selective_risk": 1.0 - acc,
            "abstention_rate": 1.0 - float(keep.mean())
        })

    return pd.DataFrame(rows)

# ============================================================
# 14. Figures
# ============================================================

def save_confusion_matrix(outputs, title, out_path):
    probs = F.softmax(outputs["target_logits"], dim=1).numpy()
    labels = outputs["labels"]
    pred = probs.argmax(axis=1)

    cm = confusion_matrix(labels, pred, labels=list(range(len(classes))))

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticklabels(classes)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.close()

def save_reliability_diagram(outputs, title, out_path, n_bins=15):
    probs = F.softmax(outputs["target_logits"], dim=1).numpy()
    labels = outputs["labels"]

    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)

    centers = []
    accs = []

    for i in range(n_bins):
        low = bins[i]
        high = bins[i + 1]
        mask = (conf > low) & (conf <= high)

        if mask.sum() > 0:
            centers.append((low + high) / 2)
            accs.append(correct[mask].mean())

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.bar(centers, accs, width=1/n_bins, edgecolor="black", alpha=0.75)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.close()

def save_selective_curve(selective_df, protocol_name, method_name, out_path):
    tmp = selective_df[
        (selective_df["protocol"] == protocol_name) &
        (selective_df["method"] == method_name)
    ].copy()

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(tmp["coverage"], tmp["selective_accuracy"], marker="o")

    ax.set_xlabel("Coverage")
    ax.set_ylabel("Selective accuracy")
    ax.set_title(f"{protocol_name}: Selective Prediction")
    ax.grid(True, linewidth=0.3)

    plt.tight_layout()
    plt.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.close()

# ============================================================
# 15. Main experiment runner
# ============================================================

all_main = []
all_per_illum = []
all_selective = []
all_coverage = []

final_method = "Proposed + Illumination-Specific Self-Calibration"

for protocol_name, (train_df, val_df, test_df) in protocols.items():

    print("\n" + "=" * 90)
    print("Running protocol:", protocol_name)
    print("=" * 90)

    train_loader = make_loader(train_df, train=True)
    val_loader = make_loader(val_df, train=False)
    test_loader = make_loader(test_df, train=False)

    model = SelfCalibratingMobileNetV3(
        num_classes=len(classes),
        num_illums=len(illums)
    ).to(DEVICE)

    print("Trainable parameters:", count_params(model))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_val_f1 = -1.0
    best_path = MODEL_DIR / f"{protocol_name}_best_model.pt"

    history = []

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, epoch)
        scheduler.step()

        val_outputs = collect_outputs(model, val_loader)

        val_probs = F.softmax(val_outputs["target_logits"], dim=1).numpy()
        val_metrics = classification_metrics(val_probs, val_outputs["labels"])

        illum_probs = F.softmax(val_outputs["illum_logits"], dim=1).numpy()
        illum_acc = accuracy_score(
            val_outputs["illums"],
            illum_probs.argmax(axis=1)
        )

        print(
            f"Epoch {epoch}/{EPOCHS} completed | "
            f"train_loss={train_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_f1={val_metrics['f1_macro']:.4f} | "
            f"val_ece={val_metrics['ece']:.4f} | "
            f"illum_acc={illum_acc:.4f}"
        )

        history.append({
            "protocol": protocol_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_illumination_accuracy": illum_acc,
            **{f"val_{k}": v for k, v in val_metrics.items()}
        })

        if val_metrics["f1_macro"] > best_val_f1:
            best_val_f1 = val_metrics["f1_macro"]
            torch.save(model.state_dict(), best_path)

        torch.cuda.empty_cache()

    pd.DataFrame(history).to_csv(
        TABLE_DIR / f"{protocol_name}_training_history.csv",
        index=False
    )

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))

    val_outputs = collect_outputs(model, val_loader)
    test_outputs = collect_outputs(model, test_loader)

    # Uncalibrated proposed model
    test_probs = F.softmax(test_outputs["target_logits"], dim=1).numpy()
    uncal_metrics = classification_metrics(test_probs, test_outputs["labels"])

    illum_probs = F.softmax(test_outputs["illum_logits"], dim=1).numpy()
    illum_acc_test = accuracy_score(
        test_outputs["illums"],
        illum_probs.argmax(axis=1)
    )

    uncal_metrics["illumination_accuracy"] = illum_acc_test

    # Illumination-specific calibration
    global_T, temp_map = fit_illumination_temperatures(val_outputs)

    calibrated_outputs = apply_illumination_temperature(
        test_outputs,
        temp_map,
        global_T
    )

    calibrated_probs = F.softmax(calibrated_outputs["target_logits"], dim=1).numpy()
    cal_metrics = classification_metrics(
        calibrated_probs,
        calibrated_outputs["labels"]
    )

    cal_metrics["illumination_accuracy"] = illum_acc_test

    main_df = pd.DataFrame([
        {
            "protocol": protocol_name,
            "method": "Proposed Self-Calibrating MobileNetV3-Small",
            "temperature": 1.0,
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df),
            "parameters": count_params(model),
            **uncal_metrics
        },
        {
            "protocol": protocol_name,
            "method": final_method,
            "temperature": "per-illumination",
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df),
            "parameters": count_params(model),
            **cal_metrics
        }
    ])

    all_main.append(main_df)

    per_df = per_illumination_metrics(
        calibrated_outputs,
        protocol_name,
        final_method
    )

    sel_df = selective_prediction_table(
        calibrated_outputs,
        protocol_name,
        final_method
    )

    cov_df = coverage_control_table(
        calibrated_outputs,
        protocol_name,
        final_method
    )

    all_per_illum.append(per_df)
    all_selective.append(sel_df)
    all_coverage.append(cov_df)

    # Save predictions
    pred = calibrated_probs.argmax(axis=1)

    reliability, target_conf, illum_conf, entropy = compute_reliability(
        calibrated_outputs
    )

    pred_df = pd.DataFrame({
        "filepath": calibrated_outputs["paths"],
        "true_label": [idx_to_class[int(i)] for i in calibrated_outputs["labels"]],
        "pred_label": [idx_to_class[int(i)] for i in pred],
        "true_illumination": [idx_to_illum[int(i)] for i in calibrated_outputs["illums"]],
        "target_confidence": target_conf,
        "illumination_confidence": illum_conf,
        "entropy": entropy,
        "self_calibrating_reliability": reliability,
        "correct": (pred == calibrated_outputs["labels"]).astype(int)
    })

    pred_df.to_csv(
        TABLE_DIR / f"{protocol_name}_predictions.csv",
        index=False
    )

    # Figures
    save_confusion_matrix(
        calibrated_outputs,
        f"{protocol_name}: Self-Calibrated Confusion Matrix",
        FIG_DIR / f"{protocol_name}_confusion_matrix_600dpi.png"
    )

    save_reliability_diagram(
        calibrated_outputs,
        f"{protocol_name}: Reliability Diagram",
        FIG_DIR / f"{protocol_name}_reliability_diagram_600dpi.png"
    )

    save_selective_curve(
        sel_df,
        protocol_name,
        final_method,
        FIG_DIR / f"{protocol_name}_selective_prediction_curve_600dpi.png"
    )

    print("Global temperature:", global_T)
    print("Illumination-specific temperatures:", {idx_to_illum[k]: v for k, v in temp_map.items()})

    torch.cuda.empty_cache()

# ============================================================
# 16. Combine and save final results
# ============================================================

main_results = pd.concat(all_main, ignore_index=True)
per_illum_results = pd.concat(all_per_illum, ignore_index=True)
selective_results = pd.concat(all_selective, ignore_index=True)
coverage_results = pd.concat(all_coverage, ignore_index=True)

main_results = add_protocol_columns(main_results)
per_illum_results = add_protocol_columns(per_illum_results)
selective_results = add_protocol_columns(selective_results)
coverage_results = add_protocol_columns(coverage_results)

main_results.to_csv(
    TABLE_DIR / "Table_1_Main_Performance_UAV_6G.csv",
    index=False
)

per_illum_results.to_csv(
    TABLE_DIR / "Table_2_Per_Illumination_Robustness_UAV_6G.csv",
    index=False
)

selective_results.to_csv(
    TABLE_DIR / "Table_3_Safe_Abstention_UAV_6G.csv",
    index=False
)

coverage_results.to_csv(
    TABLE_DIR / "Table_4_Coverage_Control_UAV_6G.csv",
    index=False
)

excel_path = OUTPUT_DIR / "Self_Calibrating_UAV_6G_Robustness_Results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    main_results.to_excel(writer, sheet_name="Main_Performance", index=False)
    per_illum_results.to_excel(writer, sheet_name="Per_Illumination", index=False)
    selective_results.to_excel(writer, sheet_name="Safe_Abstention", index=False)
    coverage_results.to_excel(writer, sheet_name="Coverage_Control", index=False)
    df.to_excel(writer, sheet_name="Dataset_Manifest", index=False)

zip_path = "/kaggle/working/self_calibrating_uav_6g_results.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    base_name="/kaggle/working/self_calibrating_uav_6g_results",
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

# ============================================================
# 17. Display final outputs
# ============================================================

print("\n" + "=" * 90)
print("DONE: UAV 5G/6G SELF-CALIBRATING ROBUSTNESS RESULTS GENERATED")
print("=" * 90)

print("Output folder:", OUTPUT_DIR)
print("Excel file:", excel_path)
print("Figures folder:", FIG_DIR)
print("Tables folder:", TABLE_DIR)
print("Zip file:", zip_path)

print("\nTABLE 1: MAIN PERFORMANCE")
display(main_results)

print("\nTABLE 2: PER-ILLUMINATION ROBUSTNESS")
display(per_illum_results)

print("\nTABLE 3: SAFE ABSTENTION")
display(selective_results)

print("\nTABLE 4: COVERAGE CONTROL")
display(coverage_results)

print("\nGenerated 600 dpi figures:")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f)